# 재활용품 객체탐지 — 실험 결과 종합

각 노트북이 남긴 리포트를 한자리에서 읽어 **숫자표 + 그래프**로 정리한다.
학습은 다시 돌리지 않는다. 각 노트북이 저장한 CSV/JSON 만 읽는다.

## 실험 계보

| # | 노트북 | 데이터 | 클래스 | 물어본 것 | 상태 |
|---|--------|--------|--------|-----------|------|
| 00 | `00_yolo26n_baseline_no_aug` | clean | **86** | 아무것도 안 하면 몇 점인가 | 완료 |
| 01 | `01_yolo_optuna_RESUME_TOTAL25` | clean | **86** | 하이퍼파라미터를 맞추면 얼마나 오르나 | 완료 |
| 02 | `02_yolo_baseline_augmentation_search` | clean | **17** | 어떤 증강이 제일 좋은가 + 2x2 대조 | 완료 (37개 실험) |
| 04 | `04_yolo_damaged_data_compare` | 파손 포함 **2배** | 17 | 파손 데이터를 넣으면 어떻게 되나 | 완료 (5가지) |
| 05 | `05_yolo_retune_17class_optuna_augmentation` | 파손 포함 2배 | 17 | 현재 데이터로 다시 튜닝하면 오르나 | 예정 (12가지) |
| 06 | `06_yolo_damaged_data_two_variants` | clean / 20% 교체 | 17 | 파손 비율이 성능을 얼마나 바꾸나 | 예정 (8가지 x 2회) |

옛 03번 노트북(`03_yolo_no_aug_tuned_B03`)은 **02번에 병합**되어
`B03` 실험으로 들어갔습니다. 원본은 `notebooks/_archive/`에 있습니다.

## ⚠ 읽을 때의 전제

**00·01 과 02 이후의 숫자를 나란히 놓고 비교하면 안 된다.**
00·01 은 세분류 **86 클래스**, 02 이후는 대분류로 병합한 **17 클래스** 다.
02 의 mAP 가 높은 것은 증강 덕이 아니라 클래스를 합쳤기 때문이다.
비교는 **같은 클래스 체계 안에서만** 한다.

또한 02(clean) 와 04(파손 포함) 는 **학습 데이터 자체가 다르므로**
같은 ID끼리 보더라도 조건 차이를 반드시 함께 적는다.


## 0. 준비

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

# ── 경로 ─────────────────────────────────────────────────────────────
NB_DIR    = Path.cwd()
REPO_ROOT = NB_DIR.parent.parent if NB_DIR.name == "notebooks" else Path.cwd()
AI        = REPO_ROOT / "ai"

P_BASE_00   = AI / "models/yolo/00_yolo_baseline_no_aug/report/baseline_summary.csv"
P_OPTUNA    = AI / "models/yolo/01_yolo_optuna_no_aug/report"
P_AUG_02    = AI / "models/yolo/02_experiment_augmentation/report/summary"
# 옛 03번(B03)은 02번에 병합되어, 아래 두 경로는 같은 폴더를 가리킨다.
P_TUNED_03  = AI / "models/yolo/02_experiment_augmentation/report/summary"

FIG_DIR = AI / "models/yolo/_summary_report/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for p in [P_BASE_00, P_OPTUNA, P_AUG_02, P_TUNED_03]:
    print(("OK  " if p.exists() else "없음 ") + str(p.relative_to(REPO_ROOT)))
print("\n그림 저장 위치:", FIG_DIR.relative_to(REPO_ROOT))

In [ ]:
# ── 표기 규칙 ────────────────────────────────────────────────────────
# 색은 카테고리 식별용으로만 쓰고, 순위를 색으로 표현하지 않는다.
# 대비가 낮은 색(aqua)이 섞이므로 막대에는 항상 값 레이블을 직접 붙인다.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
YELLOW, MAGENTA    = "#eda100", "#e87ba4"
INK, INK2, MUTED   = "#0b0b0b", "#52514e", "#8a8985"
SURFACE, GRID      = "#fcfcfb", "#e4e3df"

plt.rcParams.update({
    "font.family":       "Malgun Gothic",   # Windows 한글
    "axes.unicode_minus": False,
    "figure.facecolor":  SURFACE,
    "axes.facecolor":    SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor":    GRID,
    "axes.labelcolor":   INK2,
    "text.color":        INK,
    "xtick.color":       INK2,
    "ytick.color":       INK2,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   INK,
    "font.size":         10,
    "figure.dpi":        110,
    "savefig.dpi":       200,
    "savefig.bbox":      "tight",
})

def tidy(ax, xgrid=False):
    """축을 뒤로 물린다 — 데이터가 앞, 눈금선이 뒤."""
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.grid(axis="x" if xgrid else "y", color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    return ax

def save(fig, name):
    out = FIG_DIR / f"{name}.png"
    fig.savefig(out)
    print("저장:", out.name)
    return out

---
## 1. 86 클래스 — Optuna 하이퍼파라미터 최적화

00 번(튜닝 없음)과 01 번(Optuna) 은 **같은 데이터, 같은 86 클래스, 같은 15 epoch** 이다.
그래서 이 둘은 직접 비교해도 된다. 이 실험의 핵심 성과다.

In [ ]:
base00 = pd.read_csv(P_BASE_00, encoding="utf-8-sig").iloc[0]
sel    = json.loads((P_OPTUNA / "selected_trial.json").read_text(encoding="utf-8"))
m_opt  = sel["metrics"]

METRICS = ["precision", "recall", "f1", "mAP50", "mAP50_95"]
LABELS  = ["Precision", "Recall", "F1", "mAP50", "mAP50-95"]

cmp86 = pd.DataFrame({
    "지표":            LABELS,
    "00 baseline":     [float(base00[m]) for m in METRICS],
    "01 Optuna tuned": [float(m_opt[m])  for m in METRICS],
})
cmp86["개선폭"]  = cmp86["01 Optuna tuned"] - cmp86["00 baseline"]
cmp86["개선율%"] = cmp86["개선폭"] / cmp86["00 baseline"] * 100

display(cmp86.round(4).style.format({
    "00 baseline": "{:.4f}", "01 Optuna tuned": "{:.4f}",
    "개선폭": "{:+.4f}", "개선율%": "{:+.1f}%"}).hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.2))
x, w = np.arange(len(LABELS)), 0.36

b1 = ax.bar(x - w/2, cmp86["00 baseline"],     w, label="00 baseline (튜닝 없음)",
            color=MUTED, zorder=3)
b2 = ax.bar(x + w/2, cmp86["01 Optuna tuned"], w, label="01 Optuna tuned",
            color=BLUE, zorder=3)

for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9, color=INK2)

# 개선폭을 막대 사이에 직접 표기 — 색이 아니라 숫자로 읽히게
for xi, d in zip(x, cmp86["개선폭"]):
    ax.annotate(f"+{d:.3f}", (xi, max(cmp86.loc[xi, "00 baseline"],
                                      cmp86.loc[xi, "01 Optuna tuned"]) + 0.055),
                ha="center", fontsize=9, color=BLUE, fontweight="bold")

ax.set_xticks(x, LABELS)
ax.set_ylim(0, 0.80)
ax.set_ylabel("점수")
ax.set_title("86 클래스 · 15 epoch — Optuna 전후 (모든 지표 상승)")
ax.legend(frameon=False, loc="upper left", fontsize=9)
tidy(ax)
save(fig, "01_optuna_86class_before_after")
plt.show()

In [ ]:
trials = pd.read_csv(P_OPTUNA / "trials.csv")
done   = trials[trials.state == "COMPLETE"]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.2),
                               gridspec_kw={"width_ratios": [1.35, 1]})

# ── 왼쪽: trial 진행 ──
axL.scatter(done.trial, done.mAP50_95, s=46, color=BLUE, zorder=3,
            edgecolor=SURFACE, linewidth=1.4, label="완료한 trial")
run_best = done.sort_values("trial").mAP50_95.cummax()
axL.plot(done.sort_values("trial").trial, run_best, color=ORANGE, lw=2,
         zorder=4, label="누적 최고값")

sel_t = sel["selected_trial"]
sy = float(trials.loc[trials.trial == sel_t, "mAP50_95"].iloc[0])
axL.scatter([sel_t], [sy], s=170, facecolor="none", edgecolor=ORANGE,
            linewidth=2.4, zorder=5)
axL.annotate(f"채택: trial {sel_t}\n{sy:.4f}", (sel_t, sy), (-14, -42),
             textcoords="offset points", ha="center", fontsize=9,
             color=ORANGE, fontweight="bold")

axL.set_xlabel("trial 번호"); axL.set_ylabel("mAP50-95")
axL.set_title(f"탐색 경과 — 완료 {len(done)} / 전체 {len(trials)} trial")
axL.legend(frameon=False, fontsize=9, loc="lower right")
tidy(axL)

# ── 오른쪽: 파라미터 중요도 ──
imp = pd.read_csv(P_OPTUNA / "parameter_importance.csv").sort_values("importance")
bars = axR.barh(imp.parameter, imp.importance, color=BLUE, height=0.62, zorder=3)
axR.bar_label(bars, fmt="%.3f", padding=4, fontsize=9, color=INK2)
axR.set_xlim(0, imp.importance.max() * 1.22)
axR.set_xlabel("중요도")
axR.set_title("무엇이 성능을 좌우했나")
tidy(axR, xgrid=True)

fig.suptitle("01 · Optuna 탐색 (86 클래스)", fontsize=13, fontweight="bold", y=1.02)
save(fig, "02_optuna_search")
plt.show()

print("채택된 하이퍼파라미터")
for k, v in sel["params"].items():
    print(f"  {k:15} {v}")
print(f"\n총 탐색 시간: {trials.train_minutes.sum()/60:.1f} 시간")

In [ ]:
ms = pd.read_csv(P_OPTUNA / "final_multiseed.csv")

fig, ax = plt.subplots(figsize=(6.6, 3.6))
xs = np.arange(len(ms))
bars = ax.bar(xs, ms.mAP50_95, 0.5, color=BLUE, zorder=3)
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9, color=INK2)

mean = ms.mAP50_95.mean()
ax.axhline(mean, color=ORANGE, lw=1.8, ls="--", zorder=4)
ax.annotate(f"평균 {mean:.4f}  (표준편차 {ms.mAP50_95.std():.4f})",
            (len(ms) - 0.5, mean), (0, 8), textcoords="offset points",
            ha="right", fontsize=9, color=ORANGE, fontweight="bold")

ax.set_xticks(xs, [f"seed {s}" for s in ms.seed])
ax.set_ylim(0, 0.62); ax.set_ylabel("mAP50-95")
ax.set_title("재현성 — 시드를 바꿔도 같은 점수")
tidy(ax)
save(fig, "03_optuna_multiseed")
plt.show()

---
## 2. 17 클래스 — 증강 기법 36종 비교

02 번은 대분류로 병합한 17 클래스에서 증강 기법을 바꿔가며 36 회 학습했다 (각 10 epoch).
기준선은 `B01` (YOLO 기본 증강) 이다.

In [ ]:
rank = pd.read_csv(P_AUG_02 / "seed42_ranking.csv", encoding="utf-8-sig")
rank = rank.sort_values("mAP50_95", ascending=False).reset_index(drop=True)

view = rank[["id", "name", "family", "precision", "recall",
             "mAP50", "mAP50_95", "mAP50_95_delta_vs_B01"]]
print(f"총 {len(rank)} 개 실험 · 상위 8")
display(view.head(8).round(4))
print("하위 6 — 기하 변형을 세게 걸면 크게 무너진다")
display(view.tail(6).round(4))

In [ ]:
# 색은 '계열'이라는 정체성에만 쓴다. 순위는 위치로 읽는다.
FAMILY_COLOR = {
    "baseline":      MUTED,
    "opencv_single": BLUE,
    "opencv_combo":  AQUA,
    "yolo_single":   ORANGE,
    "yolo_combo":    YELLOW,
    "hybrid":        MAGENTA,
}
r = rank.sort_values("mAP50_95_delta_vs_B01")
colors = [FAMILY_COLOR.get(f, MUTED) for f in r.family]

fig, ax = plt.subplots(figsize=(9.6, 10.2))
ys = np.arange(len(r))
ax.barh(ys, r.mAP50_95_delta_vs_B01, color=colors, height=0.68, zorder=3)
ax.axvline(0, color=INK2, lw=1.2, zorder=4)

for y, (d, i) in enumerate(zip(r.mAP50_95_delta_vs_B01, r.id)):
    off = 0.004 if d >= 0 else -0.004
    ax.text(d + off, y, f"{d:+.3f}", va="center",
            ha="left" if d >= 0 else "right", fontsize=8, color=INK2)

ax.set_yticks(ys, [f"{i}  {n}" for i, n in zip(r.id, r.name)], fontsize=8.5)
ax.set_xlabel("mAP50-95 변화량 (기준: B01 = YOLO 기본 증강)")
ax.set_xlim(-0.26, 0.05)
ax.set_title("17 클래스 · 10 epoch — 증강 기법 36종 (seed 42)")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLOR.values()]
ax.legend(handles, FAMILY_COLOR.keys(), frameon=False, fontsize=9,
          loc="upper left", title="계열", title_fontsize=9)
tidy(ax, xgrid=True)
save(fig, "04_augmentation_ranking")
plt.show()

발표 슬라이드용 — 위아래 극단만 크게 뽑은 축약판.

In [ ]:
# 36개를 한 장에 넣으면 투사 화면에서 글자가 안 보인다.
# 발표용으로는 양 극단 6개씩만 큰 글씨로 뽑는다. 전체는 위 그림에 있다.
top6 = rank.nlargest(6, "mAP50_95_delta_vs_B01")
bot6 = rank.nsmallest(6, "mAP50_95_delta_vs_B01")
pick = pd.concat([bot6, top6]).sort_values("mAP50_95_delta_vs_B01")

fig, ax = plt.subplots(figsize=(9.0, 5.4))
ys = np.arange(len(pick))
cols = [AQUA if v >= 0 else ORANGE for v in pick.mAP50_95_delta_vs_B01]
ax.barh(ys, pick.mAP50_95_delta_vs_B01, color=cols, height=0.66, zorder=3)
ax.axvline(0, color=INK2, lw=1.4, zorder=4)

for y, d in zip(ys, pick.mAP50_95_delta_vs_B01):
    off = 0.006 if d >= 0 else -0.006
    ax.text(d + off, y, f"{d:+.3f}", va="center",
            ha="left" if d >= 0 else "right",
            fontsize=12, color=INK2, fontweight="bold")

ax.set_yticks(ys, [f"{i}  {n}" for i, n in zip(pick.id, pick.name)], fontsize=12)
ax.set_xlim(-0.27, 0.06)
ax.set_xlabel("mAP50-95 변화량 (기준: B01 = YOLO 기본 증강)", fontsize=12)
ax.tick_params(axis="x", labelsize=11)
ax.set_title("증강 기법 36종 중 상·하위 6개씩", fontsize=15)
ax.text(-0.115, 5.5, "── 나머지 24종은 이 사이 ──", fontsize=11,
        color=MUTED, ha="center", va="center")
tidy(ax, xgrid=True)
save(fig, "08_augmentation_extremes")
plt.show()

### 그런데 이 순위는 믿을 수 있나

상위 3 개를 시드 3 개(42/123/777)로 다시 돌린 결과를 보면 판단이 달라진다.

In [ ]:
msa = pd.read_csv(P_AUG_02 / "multiseed_summary.csv", encoding="utf-8-sig")
msa = msa.sort_values("mAP50_95_mean")

# 막대가 아니라 점 + 오차막대로 그린다. 길이로 크기를 읽는 그림이 아니므로
# y축을 확대해도 왜곡이 없고, 오차 구간이 겹치는 것이 그대로 보인다.
fig, ax = plt.subplots(figsize=(8.2, 3.6))
ys = np.arange(len(msa))

lo = (msa.mAP50_95_mean - msa.mAP50_95_std).max()
hi = (msa.mAP50_95_mean + msa.mAP50_95_std).min()
ax.axvspan(lo, hi, color=ORANGE, alpha=0.12, zorder=1,
           label="세 기법의 오차 구간이 모두 겹치는 범위")

ax.errorbar(msa.mAP50_95_mean, ys, xerr=msa.mAP50_95_std, fmt="o",
            color=BLUE, markersize=11, markeredgecolor=SURFACE,
            markeredgewidth=1.6, ecolor=INK2, elinewidth=1.8,
            capsize=7, capthick=1.8, zorder=4)

for y, (m, sd) in enumerate(zip(msa.mAP50_95_mean, msa.mAP50_95_std)):
    ax.text(m + sd + 0.004, y, f"{m:.4f} ± {sd:.4f}", va="center",
            fontsize=9.5, color=INK2)

ax.set_yticks(ys, [f"{i}" + chr(10) + f"{n}" for i, n in zip(msa.id, msa.name)], fontsize=9)
ax.set_ylim(-0.6, len(msa) - 0.4)
ax.set_xlim(0.74, 0.875)
ax.set_xlabel("mAP50-95 (3 시드 평균 ± 표준편차)")
ax.set_title("상위 3 기법 — 평균 차이 0.004 vs 시드 편차 ±0.02")
ax.legend(frameon=False, fontsize=9, loc="lower right")
tidy(ax, xgrid=True)
save(fig, "05_augmentation_noise")
plt.show()

spread = msa.mAP50_95_mean.max() - msa.mAP50_95_mean.min()
print(f"상위 3 간 평균 차이 : {spread:.4f}")
print(f"시드 표준편차 최대  : {msa.mAP50_95_std.max():.4f}  ← 차이보다 크다")

---
## 3. 17 클래스 — 증강과 하이퍼파라미터를 분리한 2×2

03 번이 마지막 칸(B03)을 채워 2×2 요인 설계가 완성됐다. 모두 15 epoch, seed 42.

|              | optimizer auto | Optuna tuned |
|--------------|----------------|--------------|
| 증강 OFF     | B00            | **B03**      |
| YOLO 기본 증강 | B01          | B02          |

In [ ]:
tbl = pd.read_csv(P_TUNED_03 / "hyperparameter_effect_2x2.csv", encoding="utf-8-sig")
g   = tbl.set_index("id")["mAP50_95"].astype(float)
B00, B01, B02, B03 = (g[k] for k in ["B00", "B01", "B02", "B03"])

cell = pd.DataFrame(
    [[B00, B03], [B01, B02]],
    index=["증강 OFF", "YOLO 기본 증강"],
    columns=["optimizer auto", "Optuna tuned"],
)
print("mAP50-95 (17 클래스 · 15 epoch · seed 42)")
display(cell.round(4))

effects = pd.DataFrame({
    "효과": ["증강 효과 (auto 조건)", "증강 효과 (tuned 조건)",
             "하이퍼파라미터 효과 (증강 OFF)", "하이퍼파라미터 효과 (증강 ON)"],
    "계산": ["B01 − B00", "B02 − B03", "B03 − B00", "B02 − B01"],
    "값":   [B01 - B00, B02 - B03, B03 - B00, B02 - B01],
})
display(effects.round(4))

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.0, 4.4),
                               gridspec_kw={"width_ratios": [1.1, 1]})

# ── 왼쪽: 2×2 막대 ──
x, w = np.arange(2), 0.36
auto  = [B00, B01]
tuned = [B03, B02]
b1 = axL.bar(x - w/2, auto,  w, label="optimizer auto", color=MUTED,  zorder=3)
b2 = axL.bar(x + w/2, tuned, w, label="Optuna tuned",   color=BLUE,   zorder=3)
for bars in (b1, b2):
    axL.bar_label(bars, fmt="%.4f", padding=3, fontsize=9, color=INK2)
for xi, (idl, idr) in zip(x, [("B00", "B03"), ("B01", "B02")]):
    axL.text(xi - w/2, 0.03, idl, ha="center", fontsize=9, color="white", fontweight="bold")
    axL.text(xi + w/2, 0.03, idr, ha="center", fontsize=9, color="white", fontweight="bold")

axL.set_xticks(x, ["증강 OFF", "YOLO 기본 증강"])
axL.set_ylim(0, 1.06); axL.set_ylabel("mAP50-95")
axL.set_title("2×2 요인 설계 (17 클래스 · 15 epoch)")
axL.legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
tidy(axL)

# ── 오른쪽: 효과 분해 ──
lab = ["증강 효과\n(auto)", "증강 효과\n(tuned)",
       "HP 효과\n(증강 OFF)", "HP 효과\n(증강 ON)"]
val = [B01 - B00, B02 - B03, B03 - B00, B02 - B01]
col = [AQUA if v >= 0 else ORANGE for v in val]

bars = axR.bar(np.arange(4), val, 0.52, color=col, zorder=3)
axR.axhline(0, color=INK2, lw=1.2, zorder=4)
for xi, v in zip(np.arange(4), val):
    axR.text(xi, v + (0.003 if v >= 0 else -0.003), f"{v:+.4f}",
             ha="center", va="bottom" if v >= 0 else "top",
             fontsize=9.5, color=INK2, fontweight="bold")

axR.set_xticks(np.arange(4), lab, fontsize=9)
axR.set_ylim(-0.062, 0.062); axR.set_ylabel("mAP50-95 변화량")
axR.set_title("효과 분해 — HP 튜닝이 17 클래스에선 손해")
tidy(axR)

save(fig, "06_two_by_two")
plt.show()

### 여기서 나온 가장 중요한 발견

`B03 − B00 = -0.041` — **86 클래스에서 찾은 하이퍼파라미터를 17 클래스에 그대로 쓰면 오히려 성능이 떨어진다.**

같은 하이퍼파라미터가 86 클래스에서는 `+0.144` 였다. 부호가 반대다.
증강을 켜면 `B02 − B01 = -0.002` 로 손해가 거의 사라지는데,
이는 증강이 잘못 맞춰진 학습률 스케줄을 상쇄해 준다는 뜻으로 읽힌다.

**하이퍼파라미터는 데이터셋에 종속된다.** 클래스 체계를 바꿨으면 다시 튜닝해야 한다.

---
## 4. 한 장 요약

In [ ]:
summary = pd.DataFrame([
    ["00", "86", "증강 OFF + auto",        15, float(base00.mAP50_95), "기준점"],
    ["01", "86", "증강 OFF + Optuna",      15, float(m_opt["mAP50_95"]), "+0.144 (+38%)"],
    ["02", "17", "B01 기본증강 + auto",     15, float(B01), "17클래스 최고"],
    ["02", "17", "B02 기본증강 + Optuna",   15, float(B02), "-0.002"],
    ["02", "17", "B00 증강OFF + auto",      15, float(B00), "-0.008"],
    ["03", "17", "B03 증강OFF + Optuna",    15, float(B03), "-0.041 ← HP 역효과"],
    ["02", "17", "C11 최고 증강 (10ep)",    10, float(rank.iloc[0].mAP50_95), "+0.013, 노이즈 범위"],
], columns=["노트북", "클래스", "설정", "epoch", "mAP50-95", "비고"])

display(summary.round(4).style.format({"mAP50-95": "{:.4f}"}).hide(axis="index"))
summary.to_csv(FIG_DIR.parent / "summary_table.csv", index=False, encoding="utf-8-sig")
print("\n저장:", (FIG_DIR.parent / "summary_table.csv").name)

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.8, 4.2),
                               gridspec_kw={"width_ratios": [1, 1.25]})

# 86 클래스 — 같은 축에서만 비교
axL.bar([0, 1], [float(base00.mAP50_95), float(m_opt["mAP50_95"])], 0.5,
        color=[MUTED, BLUE], zorder=3)
for xi, v in zip([0, 1], [float(base00.mAP50_95), float(m_opt["mAP50_95"])]):
    axL.text(xi, v + 0.012, f"{v:.4f}", ha="center", fontsize=10,
             color=INK, fontweight="bold")
axL.annotate("", (1, 0.60), (0, 0.60),
             arrowprops=dict(arrowstyle="->", color=BLUE, lw=2))
axL.text(0.5, 0.618, "+0.144  (+38%)", ha="center", fontsize=10,
         color=BLUE, fontweight="bold")
axL.set_xticks([0, 1], ["00 baseline", "01 Optuna"])
axL.set_ylim(0, 0.76); axL.set_ylabel("mAP50-95")
axL.set_title("86 클래스 — Optuna 는 확실히 효과 있음")
tidy(axL)

# 17 클래스 — 네 조건
ids  = ["B00", "B01", "B02", "B03"]
vals = [B00, B01, B02, B03]
cols = [MUTED, BLUE, MUTED, ORANGE]
axR.bar(np.arange(4), vals, 0.52, color=cols, zorder=3)
for xi, v in zip(np.arange(4), vals):
    axR.text(xi, v + 0.012, f"{v:.4f}", ha="center", fontsize=10, color=INK)
axR.set_xticks(np.arange(4),
               ["B00\n증강OFF\nauto", "B01\n기본증강\nauto",
                "B02\n기본증강\ntuned", "B03\n증강OFF\ntuned"], fontsize=8.5)
axR.set_ylim(0, 0.95); axR.set_ylabel("mAP50-95")
axR.set_title("17 클래스 — 손대지 않은 B01 이 최고")
tidy(axR)

fig.suptitle("클래스 체계가 다르므로 좌우 축을 서로 비교하지 말 것",
             fontsize=10, color=MUTED, y=-0.02)
save(fig, "07_headline")
plt.show()

---
## 5. 결론과 다음 단계

### 확인된 것

1. **하이퍼파라미터 최적화는 효과가 컸다 — 단, 86 클래스에서.**
   mAP50-95 `0.377 → 0.521` (+38%). 시드 3 개에서 편차 0.005 이하로 재현된다.
   중요도는 `lrf`(0.35) → `lr0`(0.22) → `warmup_epochs`(0.16) 순이고,
   `optimizer` 와 `cos_lr` 은 사실상 무의미했다.

2. **증강 기법 탐색은 소득이 거의 없었다.**
   36 종 중 최고인 `C11 opencv_horizontal_flip` 이 `+0.013`.
   그런데 상위 3 개의 평균 차이는 `0.004` 인 반면 시드 표준편차가 `±0.019~0.026` 이다.
   **순위가 노이즈에 묻혀 있다.** 15 epoch 최종 비교에서는 손대지 않은 `B01` 이 1 위였다.

3. **증강을 겹쳐 쌓을수록 나빠졌다.**
   하위권은 전부 combo·hybrid 계열이다 (`H03 -0.220`, `C13 -0.188`, `C09 -0.176`).
   기하 변형을 세게 걸면 bbox 가 망가지는 것으로 보인다.

4. **하이퍼파라미터는 데이터셋에 종속된다.**
   86 클래스에서 `+0.144` 였던 같은 설정이 17 클래스에서는 `-0.041` 이다.

### 04 번을 돌리기 전에

- 04 번은 **파손 이미지가 섞인** 데이터다. 02·03 과 숫자를 나란히 놓으면
  "파손의 영향"이 아니라 "데이터가 다른 것"을 보게 된다. 04 번 안의 5 개끼리만 비교한다.
- 위 4번 발견을 감안하면, 04 번의 `tuned` 설정은 86 클래스용 하이퍼파라미터다.
  **17 클래스 + 파손 데이터에서 Optuna 를 다시 돌리는 것**이 더 나은 선택일 수 있다.
- 앞으로 비교는 **최소 3 시드 평균**으로만 한다. 단일 시드 순위는 신뢰할 수 없다.